In [ ]:
import pandas as pd

df = pd.read_csv("spectra_441.csv")

spectra_df = df.pivot_table(
    index=["h2", "p"],
    columns="wavelength",
    values="abs"
)

spectra_df = spectra_df.sort_index(axis=1)

print("Số phổ:", spectra_df.shape[0])
print("Số điểm mỗi phổ:", spectra_df.shape[1])


Số phổ: 441
Số điểm mỗi phổ: 1001


In [ ]:
X = spectra_df.values
y = spectra_df.index.to_frame().values

print("Input shape:", X.shape)
print("Output shape:", y.shape)


Input shape: (441, 1001)
Output shape: (441, 2)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


In [ ]:
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.multioutput import MultiOutputRegressor
from xgboost import XGBRegressor

models = {
    # "RandomForest": RandomForestRegressor(
    #     n_estimators=200,
    #     random_state=42,
    #     n_jobs=-1
    # ),

    "ExtraTrees": ExtraTreesRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    # "DecisionTree": DecisionTreeRegressor(
    #     random_state=42
    # ),

    # "SVR": MultiOutputRegressor(
    #     SVR(
    #         kernel="rbf",
    #         C=100,
    #         gamma="scale",
    #         epsilon=0.01
    #     )
    #  ),

    # "XGBoost": MultiOutputRegressor(
    #     XGBRegressor(
    #         n_estimators=200,
    #         learning_rate=0.05,
    #         max_depth=6,
    #         subsample=0.8,
    #         colsample_bytree=0.8,
    #         objective="reg:squarederror",
    #         random_state=42,
    #         n_jobs=-1
        #  )
    # )
}


In [ ]:
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error

trained_models = {}

results = []

for name, model in models.items():
    print(f"\n Training {name} ...")

    model.fit(X_train, y_train)

    trained_models[name] = model

    y_pred = model.predict(X_test)

    r2 = r2_score(y_test, y_pred, multioutput="uniform_average")
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    results.append([name, r2, rmse])

    print(f"{name:12s} | R2 = {r2:.4f} | RMSE = {rmse:.6f}")


 Training ExtraTrees ...
ExtraTrees   | R2 = 0.9958 | RMSE = 0.006223


In [ ]:
!apt-get install unrar -y
!unrar x "/content/100.rar" "/content/spectra_output100_1/"

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from /content/100.rar

Creating    /content/spectra_output100_1                              OK
Creating    /content/spectra_output100_1/100                          OK
Extracting  /content/spectra_output100_1/100/001_h1_0.2205_p1_1.5606.csv       1%  OK 
Extracting  /content/spectra_output100_1/100/002_h1_0.2218_p1_1.5763.csv       2%  OK 
Extracting  /content/spectra_output100_1/100/003_h1_0.2229_p1_1.7899.csv       3%  OK 
Extracting  /content/spectra_output100_1/100/004_h1_0.2232_p1_1.7746.csv       4%  OK 
Extracting  /content/spectra_output100_1/100/005_h1_0.2245_p1_1.5007.csv       5%  OK 
Extracting  /content/spectra_output100_1/100/0

In [ ]:
import pandas as pd
import numpy as np
import os
import re
from scipy.interpolate import interp1d
from sklearn.metrics import r2_score, mean_squared_error


model_name = "ExtraTrees"
model = trained_models[model_name]

data_dir = "/content/spectra_output100_1"


wl_train = spectra_df.columns.astype(float) / 1000

results = []


pattern = r"h1_([0-9.]+)_p1_([0-9.]+)"

for root, dirs, files in os.walk(data_dir):
    for file in files:
        if file.lower().endswith(".csv"):
            filepath = os.path.join(root, file)

            match = re.search(pattern, file)
            if match:
                h1_true = float(match.group(1))
                p1_true = float(match.group(2).rstrip('.'))
            else:
                print(f"❌ Không đọc được h1 p1 từ: {file}")
                continue

            try:
                df = pd.read_csv(filepath)

                wl_new  = df.iloc[:,0].values
                abs_new = df.iloc[:,1].values

                f_interp = interp1d(
                    wl_new, abs_new,
                    kind="linear",
                    fill_value="extrapolate"
                )

                abs_interp = f_interp(wl_train)
                X_new = abs_interp.reshape(1, -1)


                pred = model.predict(X_new)

                if pred.shape[1] != 2:
                    print(f"❌ Sai model tại: {file}")
                    continue

                h1_pred = pred[0,0]
                p1_pred = pred[0,1]

                results.append([h1_true, h1_pred, p1_true, p1_pred])

            except Exception as e:
                print(f"❌ Lỗi file {file}: {e}")

df_results = pd.DataFrame(results, columns=[
    "h1_true", "h1_pred",
    "p1_true", "p1_pred"
])

df_results.to_csv("inverse_results_full.csv", index=False)

print("🎯 Saved: inverse_results_full.csv")
print("📊 Tổng số mẫu:", len(df_results))



# h1
r2_h1 = r2_score(df_results["h1_true"], df_results["h1_pred"])
rmse_h1 = np.sqrt(mean_squared_error(df_results["h1_true"], df_results["h1_pred"]))

# p1
r2_p1 = r2_score(df_results["p1_true"], df_results["p1_pred"])
rmse_p1 = np.sqrt(mean_squared_error(df_results["p1_true"], df_results["p1_pred"]))

print("\n📊 Evaluation Results:")
print(f"h1 → R² = {r2_h1:.4f}, RMSE = {rmse_h1:.4f}")
print(f"p1 → R² = {r2_p1:.4f}, RMSE = {rmse_p1:.4f}")

🎯 Saved: inverse_results_full.csv
📊 Tổng số mẫu: 100

📊 Evaluation Results:
h1 → R² = 0.9443, RMSE = 0.0068
p1 → R² = 0.9352, RMSE = 0.0297


In [ ]:
import pandas as pd
import numpy as np
import os
import re
from scipy.interpolate import interp1d

model_name = "ExtraTrees"
model = trained_models[model_name]

data_dir = "/content/spectra_output100_1"

wl_train = spectra_df.columns.astype(float) / 1000

results = []

pattern = r"h1_([0-9.]+)_p1_([0-9.]+)"

for root, dirs, files in os.walk(data_dir):
    for file in files:
        if file.lower().endswith(".csv"):
            filepath = os.path.join(root, file)


            match = re.search(pattern, file)
            if match:
                h1_true = float(match.group(1))
                p1_true = float(match.group(2).rstrip('.'))
            else:
                print(f"❌ Unable to read h1 p1 from: {file}")
                continue

            try:
                df = pd.read_csv(filepath)

                wl_new  = df.iloc[:,0].values
                abs_new = df.iloc[:,1].values


                f_interp = interp1d(wl_new, abs_new,
                                    kind="linear",
                                    fill_value="extrapolate")

                abs_interp = f_interp(wl_train)
                X_new = abs_interp.reshape(1, -1)


                pred = model.predict(X_new)

                if pred.shape[1] != 2:
                    print(f"❌ Wrong model at: {file}")
                    continue

                h1_pred = pred[0,0]
                p1_pred = pred[0,1]

                results.append([h1_true, h1_pred, p1_true, p1_pred])

            except Exception as e:
                print(f"❌ File error {file}: {e}")

# ===== lưu kết quả =====
df_results = pd.DataFrame(results, columns=[
    "h1_true", "h1_pred",
    "p1_true", "p1_pred"
])

df_results.to_csv("inverse_results_full.csv", index=False)

print("🎯 Saved: inverse_results_full.csv")
print("📊 Total number of samples:", len(df_results))

🎯 Saved: inverse_results_full.csv
📊 Tổng số mẫu: 100
